In [ ]:
!pip install openai numpy

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-m3")
print("모델 로드 완료!")


In [ ]:
sentence = "강아지를 키우는 방법"
vector = model.encode(sentence)

print(f"텍스트: {sentence}")
print(f"벡터 차원 수: {vector.shape}")
print(f"벡터 앞 10개 값: {vector[:10]}")

In [ ]:
import numpy as np

sentences = [
    "강아지를 키우는 방법",      # 기준 문장
    "개를 훈련시키는 법",        # 비슷한 의미
    "고양이 밥 주는 방법",       # 조금 다름
    "자동차 엔진 수리하기",      # 전혀 다름
]

vectors = model.encode(sentences)

# 기준 문장(0번)과 나머지 문장들의 코사인 유사도 계산
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

base = vectors[0]
for i, sentence in enumerate(sentences):
    score = cosine_similarity(base, vectors[i])
    print(f"{score:.4f}  |  {sentence}")


In [ ]:
# 문서 목록 (나중에 Vector DB에 저장될 것들)
documents = [
    "강아지는 하루에 두 번 밥을 먹어야 합니다.",
    "고양이는 독립적인 동물로 혼자 있는 시간을 좋아합니다.",
    "FAISS는 Facebook AI에서 만든 벡터 검색 라이브러리입니다.",
    "파이썬은 데이터 분석에 많이 사용되는 프로그래밍 언어입니다.",
    "강아지 훈련 시 긍정적 강화 방식이 효과적입니다.",
]

query = "강아지 키울 때 밥은 얼마나 줘야 해?"

# 문서 + 질문 임베딩
doc_vectors = model.encode(documents)
query_vector = model.encode(query)

# 유사도 계산
scores = [cosine_similarity(query_vector, doc_vec) for doc_vec in doc_vectors]

# 결과 출력
ranked = sorted(zip(scores, documents), reverse=True)
print(f"질문: {query}\n")
print("=== 관련도 순위 ===")
for score, doc in ranked:
    print(f"{score:.4f}  |  {doc}")


In [ ]:
!pip install chromadb

In [ ]:
import chromadb

# 로컬 영구 저장 (데이터 유지)
client = chromadb.PersistentClient(path="./chroma_db")

# 컬렉션 생성
collection = client.create_collection("medicine_docs")

# 문서 추가
collection.add(
    documents=[
        "타이레놀은 해열 및 진통 효과가 있는 아세트아미노펜 성분의 의약품입니다.",
        "이부프로펜은 소염진통제로 두통, 치통, 근육통에 효과적입니다.",
        "항생제는 의사의 처방 없이 복용하면 내성이 생길 수 있습니다.",
    ],
    metadatas=[
        {"category": "진통제", "requires_prescription": False},
        {"category": "소염진통제", "requires_prescription": False},
        {"category": "항생제", "requires_prescription": True},
    ],
    ids=["doc1", "doc2", "doc3"]
)

# 검색
results = collection.query(
    query_texts=["두통이 심한데 어떤 약을 먹어야 하나요?"],
    n_results=2
)

print(results['documents'])

In [ ]:
# BGE-M3 임베딩 단독 테스트
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("BAAI/bge-m3")

sentences = [
    "두통이 심해요",
    "머리가 아파요",
    "배가 고파요"
]

vectors = model.encode(sentences)

score_1 = cosine_similarity([vectors[0]], [vectors[1]])[0][0]
score_2 = cosine_similarity([vectors[0]], [vectors[2]])[0][0]

print(f"'두통이 심해요' vs '머리가 아파요' : {score_1:.4f}")
print(f"'두통이 심해요' vs '배가 고파요'  : {score_2:.4f}")


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from transformers import pipeline

# BGE-M3 임베딩
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# LLM
pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=512
)
llm = HuggingFacePipeline(pipeline=pipe)


In [ ]:
from langchain_core.documents import Document
from langchain_chroma import Chroma

documents = [
    Document(
        page_content="타이레놀(아세트아미노펜)은 해열 및 진통 효과가 있습니다. 두통, 발열에 효과적이며 임산부도 복용 가능합니다. 1회 1~2정, 1일 3회 복용합니다.",
        metadata={"drug_name": "타이레놀", "category": "진통제", "requires_prescription": False, "safe_for_pregnant": True}
    ),
    Document(
        page_content="이부프로펜은 소염진통제로 두통, 치통, 근육통, 생리통에 효과적입니다. 임산부는 복용을 피해야 합니다. 1회 1정, 1일 3회 복용합니다.",
        metadata={"drug_name": "이부프로펜", "category": "소염진통제", "requires_prescription": False, "safe_for_pregnant": False}
    ),
    Document(
        page_content="게보린은 두통, 치통, 생리통에 사용되는 복합 진통제입니다. 임산부 복용 금지. 1회 1정, 1일 3회 복용합니다.",
        metadata={"drug_name": "게보린", "category": "진통제", "requires_prescription": False, "safe_for_pregnant": False}
    ),
    Document(
        page_content="아목시실린은 항생제로 세균 감염 치료에 사용됩니다. 반드시 의사의 처방이 필요합니다.",
        metadata={"drug_name": "아목시실린", "category": "항생제", "requires_prescription": True, "safe_for_pregnant": False}
    ),
]

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="medicine_rag"
)

print("✅ ChromaDB 저장 완료")

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Retriever 정의
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 2,
        "filter": {
            "$and": [
                {"requires_prescription": {"$eq": False}},
                {"safe_for_pregnant": {"$eq": True}}
            ]
        }
    }
)

# 프롬프트 템플릿
prompt_template = """
당신은 의약품 정보를 안내하는 전문 상담사입니다.
아래 참고 문서를 바탕으로 질문에 답변해주세요.
참고 문서에 없는 내용은 절대 답변하지 마세요.

참고 문서:
{context}

질문: {question}

아래 형식으로 답변해주세요:
- 약 이름:
- 주요 성분:
- 효능:
- 복용 방법:
- 주의사항:
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

# 문서 포맷 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# LCEL RAG 체인
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [ ]:
# 시나리오: 임산부 + 두통 + 처방전 없음
query = "저 임산부인데 두통이 심해요. 처방전 없이 살 수 있는 약 있나요?"
docs = retriever.invoke(query)

print(f"🔍 질문: {query}\n")
print("📄 검색된 문서:")
for i, doc in enumerate(docs):
    print(f"\n{i+1}위: {doc.page_content}")
    print(f"   메타데이터: {doc.metadata}")
